# quantum-cortex — the decode policies on the v7 checkpoint (ADR-008 amendment 2026-09-14): the closing test of step 4

**Settings:** Accelerator = None (session B runs on CPU), Internet = On. **Inputs:** the v7 notebook's version whose Output holds `runs-keep/journal-n1-v7/` (the checkpoint, the sealed journal and its payloads, `hm-lm.json`). **Secret:** `QUANTUM_CORTEX_JOURNAL_KEY` (the journal is sealed with it).

**What runs:** session B in a new process, twice on the same checkpoint and journal, with the two decode policies declared before this measurement: `head` (the matching head decides: m ≤ 0 abstains, m > 0 forces the citation and the model generates the label and the attribute) and `head+pointer` (the head decides, the label is the line with the most reader attention from the decision position, the model generates only the attribute). Each writes `metrics/mqar/hm-lm-<run_id>-session-b-<policy>.json` with the four gate conditions a new process can evaluate (gap, recall off the window, invalid citations, claims on absent entities); the skill delta is the in process arm's, printed from the v7 `hm-lm.json`. The thresholds are the frozen ones. About 10 minutes.

**Download the log as a file before reading any number.** Then commit the two policy files next to the v7 artefacts.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # the repository is private
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install jsonschema cryptography


In [ ]:
# the key: from the Kaggle secret to the environment, never to a file
import os
from kaggle_secrets import UserSecretsClient
os.environ['QUANTUM_CORTEX_JOURNAL_KEY'] = UserSecretsClient().get_secret('QUANTUM_CORTEX_JOURNAL_KEY')
assert len(bytes.fromhex(os.environ['QUANTUM_CORTEX_JOURNAL_KEY'])) == 32, 'the key must be 32 bytes hex'
print('journal key: present (not printed)')


In [ ]:
# the v7 checkpoint, its sealed journal and its in process result, from the mounted Input, copied to a writable directory
import glob, os, shutil, json
CONFIG = 'configs/kaggle_t4_journal_v7.json'
RUN = 'journal-n1-v7'
cands = sorted(glob.glob(f'/kaggle/input/**/{RUN}/ckpt.pt', recursive=True))
src = next((os.path.dirname(c) for c in cands if os.path.exists(os.path.join(os.path.dirname(c), 'journal.jsonl'))), None)
assert src, f'no Input holds {RUN}/ckpt.pt next to its journal.jsonl: mount the v7 notebook version'
dst = f'runs/{RUN}'
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree(src, dst)
print('copied', src, '->', dst, os.listdir(dst))
a = json.load(open(f'{dst}/hm-lm.json'))
print('v7 in process:', a['verdict'], '| skill delta', a['hm_skill_delta'], '| head acc on / negctrl', a.get('hm_match_acc_on'), a.get('hm_match_acc_negctrl'))


In [ ]:
!python -m cortex_c2b.hm_lm --session-b --policy head --ckpt {dst}/ckpt.pt --journal {dst}/journal.jsonl --config {CONFIG}


In [ ]:
import json, glob, os
print('v7 in process (plain policy):', a['verdict'], '| skill delta', a['hm_skill_delta'])
print('   ', {k: a.get(k) for k in ('hm_recall_on','hm_recall_off','hm_gap','hm_invalid_citation_on','hm_valid_citation_on','hm_false_abstention_on','hm_negctrl_rate','hm_attr_exact_given_valid','hm_match_acc_on','hm_match_acc_negctrl')})
for pol, suf in (('head', '-head'), ('head+pointer', '-head-pointer')):
    fs = glob.glob(f'metrics/mqar/hm-lm-*-session-b{suf}.json')
    if not fs: print(pol, ': no file'); continue
    b = json.load(open(max(fs, key=os.path.getmtime)))          # the one this session wrote
    print(pol, '|', b['verdict_without_skill_arm'])
    print('   ', {k: b.get(k) for k in ('hm_recall_on','hm_recall_off','hm_gap','hm_invalid_citation_on','hm_valid_citation_on','hm_false_abstention_on','hm_negctrl_rate','hm_attr_exact_given_valid','hm_match_acc_on','hm_match_acc_negctrl')})
print('Commit: the two policy files under metrics/mqar/. Read cold first; the thresholds are the frozen ones.')


In [ ]:
import json, glob, os
rid = a.get('run_id') or json.load(open(f'{dst}/hm-lm.json')).get('run_id')
for pol, suf in (('plain', ''), ('head', '-head'), ('head+pointer', '-head-pointer')):
    fs = glob.glob(f'metrics/mqar/hm-lm-*-session-b{suf}.json')
    fs = [f for f in fs if suf or f.endswith('-session-b.json')]
    if not fs: print(pol, ': no file'); continue
    b = json.load(open(max(fs, key=os.path.getmtime)))
    print(pol, '|', b.get('verdict_without_skill_arm', '(plain: verdict in the v7 hm-lm.json)'))
    print('   ', {k: b.get(k) for k in ('hm_recall_on','hm_recall_off','hm_gap','hm_invalid_citation_on','hm_valid_citation_on','hm_false_abstention_on','hm_negctrl_rate','hm_attr_exact_given_valid','hm_match_acc_on','hm_match_acc_negctrl')})
print('skill delta (in process, policy independent):', a['hm_skill_delta'])
print('Commit: the two policy files under metrics/mqar/. Read cold first; the thresholds are the frozen ones.')


In [ ]:
# a manifest of the artefacts with their hashes, to verify any later retrieval
import hashlib, glob, json, os
files = sorted(glob.glob('metrics/mqar/hm-lm-*-session-b-*.json'))
manifest = {f: {'bytes': os.path.getsize(f), 'sha256_16': hashlib.sha256(open(f, 'rb').read()).hexdigest()[:16]} for f in files}
print(json.dumps(manifest, indent=1))
